# 📊 Customer Segmentation using RFM

## 🧾 Introducción

Este proyecto aplica la metodología RFM para segmentar clientes 
de un negocio de retail online, con el objetivo de identificar 
clientes valiosos, clientes en riesgo y oportunidades de crecimiento.

---

## 🎯 Objetivo

- Clasificar clientes según su comportamiento de compra
- Identificar clientes VIP
- Detectar clientes inactivos o en riesgo
- Generar estrategias de negocio basadas en datos

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df = pd.read_csv('../data/online_retail.csv', encoding='ISO-8859-1')
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 69.3 MB


In [7]:
df = df.dropna(subset=['CustomerID'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [8]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [9]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,182,4310.00
12348.0,75,31,1797.24
12349.0,19,73,1757.55
12350.0,310,17,334.40


In [17]:
# Crear scores
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'], 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

# Convertir a int
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

# Crear RFM score combinado
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) + 
    rfm['F_score'].astype(str) + 
    rfm['M_score'].astype(str)
)

In [18]:
def segment_customer(row):
    if row['RFM_Score'] == '555':
        return 'VIP'
    elif row['R_score'] >= 4:
        return 'Loyal'
    elif row['R_score'] == 1:
        return 'Lost'
    else:
        return 'Regular'

In [23]:
# Crear segmento
rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Conteo
segment_counts = rfm['Segment'].value_counts()

# Gráfica
plt.figure(figsize=(8,5))
segment_counts.plot(kind='bar')

plt.title('Distribución de Clientes por Segmento')
plt.xlabel('Segmento')
plt.ylabel('Cantidad')

plt.savefig('../imagenes/Distribucion de Clientes por Segmento.png', bbox_inches='tight')

plt.tight_layout()
plt.show()

/tmp/ipykernel_2645/3262475422.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 💡 Insights

- Existe un grupo de clientes VIP con alto valor para el negocio.
- Se identifican clientes perdidos que requieren reactivación.
- La mayoría de clientes se encuentra en segmentos intermedios.

## 🚀 Recomendaciones

- Crear programas de fidelización para clientes VIP
- Implementar campañas de reactivación para clientes perdidos
- Incentivar compras recurrentes en clientes regulares

## 🎯 Conclusión

La segmentación RFM permite entender el comportamiento de los clientes 
y enfocar estrategias específicas para maximizar el valor del negocio.